В 2084 году человечество установило первый контакт с внеземной цивилизацией, обитающей на планете Зета в созвездии Андромеды. Инопланетяне, которых назвали зетанами, обладают высокоразвитой технологией и стремятся к обмену знаниями с землянами. Для успешного установления контакта и развития взаимовыгодных отношений, чрезвычайно важно наладить эффективную коммуникацию.

Зетаны предоставили человечеству обширную текстовую библиотеку на своем языке, включающую как оригинальные произведения, так и переводы известных земных текстов. В свою очередь, они получили доступ к библиотекам Земли. Однако алгоритмы машинного перевода пока плохо справляются с необычным строением языка зетан, что делает перевод неточным и неполным.

Необходимо обучить модель перевода с языка зетан на английский.

Данные: https://disk.yandex.ru/d/u8mmcUBn64p_Nw

Формат решения: json-lines в формате, аналогичной валидации, с переводами исходных текстов в тестовой выборке

Метрика качества: BLEU между переводами и английскими референсами на закрытом тесте

In [2]:
# Imports + robust environment setup for long training
import json
import math
import os
import random
import subprocess
import sys
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Optional, Tuple

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import numpy as np
import torch
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm


def ensure_package(package_name: str):
    try:
        __import__(package_name)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package_name])


for pkg in ["transformers", "sacrebleu", "sentencepiece"]:
    ensure_package(pkg)

import sacrebleu
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer, get_linear_schedule_with_warmup


def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if torch.cuda.is_available():
    total_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"GPU memory: {total_gb:.1f} GB")

# ---- Main config ----
DATA_DIR_CANDIDATES = [Path("data"), Path("task2/4/data"), Path("./task2/4/data")]

# Auto-select stronger model when memory allows.
if torch.cuda.is_available() and (torch.cuda.get_device_properties(0).total_memory / (1024**3)) >= 20:
    MODEL_NAME = "google/byt5-base"
else:
    MODEL_NAME = "google/byt5-small"

# You can force model manually, e.g.:
# MODEL_NAME = "google/byt5-small"

SOURCE_PREFIX = "translate Zetan to English: "
OUTPUT_DIR = Path("outputs_zetan_byt5_notebook")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
BEST_MODEL_DIR = OUTPUT_DIR / "best_model"
LAST_MODEL_DIR = OUTPUT_DIR / "last_model"
SUBMISSION_PATH = OUTPUT_DIR / "submission_alien_translation.jsonl"

# Lengths (based on dataset stats; 512 is safe for almost all samples)
MAX_SOURCE_LENGTH = 512
MAX_TARGET_LENGTH = 192
MAX_NEW_TOKENS = 180

# Decoding params for quality
EVAL_NUM_BEAMS = 4
FINAL_NUM_BEAMS = 8
LENGTH_PENALTY = 0.9
REPETITION_PENALTY = 1.12
NO_REPEAT_NGRAM_SIZE = 3

# Training params (overnight)
NUM_EPOCHS = 12
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.06
MAX_GRAD_NORM = 1.0
GRADIENT_ACCUMULATION_STEPS = 8
PATIENCE = 4
MIN_DELTA = 0.0

if MODEL_NAME.endswith("-base"):
    TRAIN_BATCH_SIZE = 2 if torch.cuda.is_available() else 1
    EVAL_BATCH_SIZE = 4 if torch.cuda.is_available() else 1
    PREDICT_BATCH_SIZE = 4 if torch.cuda.is_available() else 1
else:
    TRAIN_BATCH_SIZE = 4 if torch.cuda.is_available() else 1
    EVAL_BATCH_SIZE = 8 if torch.cuda.is_available() else 1
    PREDICT_BATCH_SIZE = 8 if torch.cuda.is_available() else 1

# Optional limits for debugging; keep None for real run.
MAX_TRAIN_SAMPLES = None
MAX_VAL_SAMPLES = None

# Resume support: if True and checkpoint exists in LAST_MODEL_DIR, continue from it.
RESUME_IF_POSSIBLE = True

# Optional final extra pass on train+val with lower lr (can slightly improve hidden BLEU)
FINAL_FINETUNE_ON_TRAIN_PLUS_VAL = True
FINAL_FINETUNE_EPOCHS = 1
FINAL_FINETUNE_LR = 5e-5


c:\Users\Администратор\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


KeyboardInterrupt: 

In [2]:
# Data loading

def resolve_data_dir(candidates: List[Path]) -> Path:
    for candidate in candidates:
        if (candidate / "train").exists() and (candidate / "val").exists() and (candidate / "test_no_reference").exists():
            return candidate
    raise FileNotFoundError("Could not find data directory with train/val/test_no_reference")


def load_jsonl(path: Path, has_dst: bool) -> List[Dict[str, str]]:
    rows: List[Dict[str, str]] = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            obj = json.loads(line)
            if has_dst:
                rows.append({"src": obj["src"], "dst": obj["dst"]})
            else:
                rows.append({"src": obj["src"]})
    return rows


def maybe_slice(rows: List[Dict[str, str]], limit: Optional[int]) -> List[Dict[str, str]]:
    if limit is None or limit <= 0 or limit >= len(rows):
        return rows
    return rows[:limit]


data_dir = resolve_data_dir(DATA_DIR_CANDIDATES)
train_rows = maybe_slice(load_jsonl(data_dir / "train", has_dst=True), MAX_TRAIN_SAMPLES)
val_rows = maybe_slice(load_jsonl(data_dir / "val", has_dst=True), MAX_VAL_SAMPLES)
test_rows = load_jsonl(data_dir / "test_no_reference", has_dst=False)

print(f"Data dir: {data_dir}")
print(f"Train: {len(train_rows)} | Val: {len(val_rows)} | Test: {len(test_rows)}")
print("Example src:", train_rows[0]["src"])
print("Example dst:", train_rows[0]["dst"])

# quick byte-length diagnostics
src_byte_lens = [len(x["src"].encode("utf-8")) for x in train_rows[: min(len(train_rows), 50000)]]
dst_byte_lens = [len(x["dst"].encode("utf-8")) for x in train_rows[: min(len(train_rows), 50000)]]
print(
    f"source bytes avg={np.mean(src_byte_lens):.1f}, p95={np.percentile(src_byte_lens,95):.1f}, max={np.max(src_byte_lens)}"
)
print(
    f"target bytes avg={np.mean(dst_byte_lens):.1f}, p95={np.percentile(dst_byte_lens,95):.1f}, max={np.max(dst_byte_lens)}"
)


Data dir: data
Train: 300000 | Val: 500 | Test: 1000
Example src: ◄▴◓◠▨ ◨▽◠▦◈◬◓▪▼◬▵
Example dst: - Intriguing.
source bytes avg=95.2, p95=209.0, max=849
target bytes avg=39.1, p95=90.0, max=395


In [3]:
# Datasets and collators

class PairDataset(Dataset):
    def __init__(self, rows: List[Dict[str, str]]) -> None:
        self.rows = rows

    def __len__(self) -> int:
        return len(self.rows)

    def __getitem__(self, idx: int) -> Dict[str, str]:
        return self.rows[idx]


@dataclass
class Collator:
    tokenizer: AutoTokenizer
    max_source_length: int
    max_target_length: int
    prefix: str

    def train_collate(self, batch: List[Dict[str, str]]) -> Dict[str, torch.Tensor]:
        src_texts = [self.prefix + x["src"] for x in batch]
        tgt_texts = [x["dst"] for x in batch]

        model_inputs = self.tokenizer(
            src_texts,
            truncation=True,
            max_length=self.max_source_length,
            padding=True,
            return_tensors="pt",
        )

        labels = self.tokenizer(
            text_target=tgt_texts,
            truncation=True,
            max_length=self.max_target_length,
            padding=True,
            return_tensors="pt",
        )["input_ids"]
        labels[labels == self.tokenizer.pad_token_id] = -100

        model_inputs["labels"] = labels
        return model_inputs

    def infer_collate(self, batch: List[Dict[str, str]]) -> Dict[str, List[str]]:
        src_texts = [self.prefix + x["src"] for x in batch]
        return {"src_texts": src_texts}


def move_to_device(batch: Dict[str, torch.Tensor], device: torch.device) -> Dict[str, torch.Tensor]:
    return {k: v.to(device) for k, v in batch.items()}


In [1]:
# Model, generation and BLEU helpers

def create_model_and_tokenizer(model_name: str):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
    return model, tokenizer


@torch.no_grad()
def generate_texts(
    model: AutoModelForSeq2SeqLM,
    tokenizer: AutoTokenizer,
    dataloader: DataLoader,
    device: torch.device,
    max_source_length: int,
    max_new_tokens: int,
    num_beams: int,
    length_penalty: float,
    repetition_penalty: float,
    no_repeat_ngram_size: int,
) -> List[str]:
    model.eval()
    predictions: List[str] = []

    for batch in tqdm(dataloader, desc="Generating", leave=False):
        encoded = tokenizer(
            batch["src_texts"],
            truncation=True,
            max_length=max_source_length,
            padding=True,
            return_tensors="pt",
        )
        encoded = move_to_device(encoded, device)

        generated = model.generate(
            **encoded,
            max_new_tokens=max_new_tokens,
            num_beams=num_beams,
            length_penalty=length_penalty,
            repetition_penalty=repetition_penalty,
            no_repeat_ngram_size=no_repeat_ngram_size,
            early_stopping=True,
        )
        texts = tokenizer.batch_decode(generated, skip_special_tokens=True)
        predictions.extend([x.strip() for x in texts])

    return predictions


@torch.no_grad()
def evaluate_bleu(
    model: AutoModelForSeq2SeqLM,
    tokenizer: AutoTokenizer,
    rows: List[Dict[str, str]],
    collator: Collator,
    device: torch.device,
    batch_size: int,
    max_source_length: int,
    max_new_tokens: int,
    num_beams: int,
    length_penalty: float,
    repetition_penalty: float,
    no_repeat_ngram_size: int,
) -> Tuple[float, List[str]]:
    loader = DataLoader(
        PairDataset(rows),
        batch_size=batch_size,
        shuffle=False,
        collate_fn=collator.infer_collate,
    )

    preds = generate_texts(
        model=model,
        tokenizer=tokenizer,
        dataloader=loader,
        device=device,
        max_source_length=max_source_length,
        max_new_tokens=max_new_tokens,
        num_beams=num_beams,
        length_penalty=length_penalty,
        repetition_penalty=repetition_penalty,
        no_repeat_ngram_size=no_repeat_ngram_size,
    )
    refs = [x["dst"] for x in rows]
    bleu = float(sacrebleu.corpus_bleu(preds, [refs]).score)
    return bleu, preds


NameError: name 'torch' is not defined

In [5]:
# Training pipeline (robust, resume-capable, with early stopping)

model_checkpoint_to_load = None
if RESUME_IF_POSSIBLE and (LAST_MODEL_DIR / "pytorch_model.bin").exists():
    model_checkpoint_to_load = LAST_MODEL_DIR

if model_checkpoint_to_load is not None:
    print(f"Resuming model from {model_checkpoint_to_load}")
    tokenizer = AutoTokenizer.from_pretrained(model_checkpoint_to_load)
    model = AutoModelForSeq2SeqLM.from_pretrained(model_checkpoint_to_load)
else:
    print(f"Starting from pretrained model: {MODEL_NAME}")
    model, tokenizer = create_model_and_tokenizer(MODEL_NAME)

if hasattr(model, "gradient_checkpointing_enable"):
    model.gradient_checkpointing_enable()
if hasattr(model.config, "use_cache"):
    model.config.use_cache = False

model = model.to(device)

collator = Collator(
    tokenizer=tokenizer,
    max_source_length=MAX_SOURCE_LENGTH,
    max_target_length=MAX_TARGET_LENGTH,
    prefix=SOURCE_PREFIX,
)

train_loader = DataLoader(
    PairDataset(train_rows),
    batch_size=TRAIN_BATCH_SIZE,
    shuffle=True,
    collate_fn=collator.train_collate,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
)

optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
steps_per_epoch = math.ceil(len(train_loader) / GRADIENT_ACCUMULATION_STEPS)
total_steps = max(1, steps_per_epoch * NUM_EPOCHS)
warmup_steps = int(total_steps * WARMUP_RATIO)
scheduler = get_linear_schedule_with_warmup(optimizer, warmup_steps, total_steps)

# Mixed precision setup
use_cuda = torch.cuda.is_available()
autocast_device = "cuda" if use_cuda else "cpu"
use_bf16 = bool(use_cuda and hasattr(torch.cuda, "is_bf16_supported") and torch.cuda.is_bf16_supported())
use_fp16 = bool(use_cuda and not use_bf16)
autocast_enabled = use_bf16 or use_fp16
autocast_dtype = torch.bfloat16 if use_bf16 else torch.float16
scaler = torch.amp.GradScaler(device=autocast_device, enabled=use_fp16)

best_bleu = -1.0
epochs_without_improvement = 0
history = []

BEST_MODEL_DIR.mkdir(parents=True, exist_ok=True)
LAST_MODEL_DIR.mkdir(parents=True, exist_ok=True)

for epoch in range(1, NUM_EPOCHS + 1):
    model.train()
    optimizer.zero_grad(set_to_none=True)

    epoch_loss = 0.0
    finite_steps = 0
    skipped_non_finite = 0

    pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{NUM_EPOCHS}")
    for step, batch in enumerate(pbar, start=1):
        batch = move_to_device(batch, device)

        with torch.amp.autocast(
            device_type=autocast_device,
            enabled=autocast_enabled,
            dtype=autocast_dtype,
        ):
            loss = model(**batch).loss / GRADIENT_ACCUMULATION_STEPS

        if not torch.isfinite(loss):
            skipped_non_finite += 1
            optimizer.zero_grad(set_to_none=True)
            pbar.set_postfix(loss="nan", skipped=skipped_non_finite)
            continue

        scaler.scale(loss).backward()
        epoch_loss += loss.item() * GRADIENT_ACCUMULATION_STEPS
        finite_steps += 1

        should_step = (step % GRADIENT_ACCUMULATION_STEPS == 0) or (step == len(train_loader))
        if should_step:
            prev_scale = scaler.get_scale() if use_fp16 else None
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)

            if use_fp16:
                step_was_applied = scaler.get_scale() >= prev_scale
            else:
                step_was_applied = True
            if step_was_applied:
                scheduler.step()

        running_loss = epoch_loss / max(1, finite_steps)
        pbar.set_postfix(loss=f"{running_loss:.4f}", skipped=skipped_non_finite, lr=f"{optimizer.param_groups[0]['lr']:.2e}")

    train_loss = epoch_loss / max(1, finite_steps)

    if hasattr(model.config, "use_cache"):
        model.config.use_cache = True

    val_bleu, val_preds = evaluate_bleu(
        model=model,
        tokenizer=tokenizer,
        rows=val_rows,
        collator=collator,
        device=device,
        batch_size=EVAL_BATCH_SIZE,
        max_source_length=MAX_SOURCE_LENGTH,
        max_new_tokens=MAX_NEW_TOKENS,
        num_beams=EVAL_NUM_BEAMS,
        length_penalty=LENGTH_PENALTY,
        repetition_penalty=REPETITION_PENALTY,
        no_repeat_ngram_size=NO_REPEAT_NGRAM_SIZE,
    )

    if hasattr(model.config, "use_cache"):
        model.config.use_cache = False

    history.append(
        {
            "epoch": epoch,
            "train_loss": float(train_loss),
            "val_bleu": float(val_bleu),
            "lr": float(optimizer.param_groups[0]["lr"]),
        }
    )

    print(f"Epoch {epoch}: train_loss={train_loss:.4f}, val_BLEU={val_bleu:.2f}")
    print("  sample val pred:", val_preds[0])
    print("  sample val gold:", val_rows[0]["dst"])

    # Always save last state to resume if needed
    model.save_pretrained(LAST_MODEL_DIR)
    tokenizer.save_pretrained(LAST_MODEL_DIR)
    with (OUTPUT_DIR / "history_last.json").open("w", encoding="utf-8") as f:
        json.dump(history, f, ensure_ascii=False, indent=2)

    if val_bleu > best_bleu + MIN_DELTA:
        best_bleu = val_bleu
        epochs_without_improvement = 0
        model.save_pretrained(BEST_MODEL_DIR)
        tokenizer.save_pretrained(BEST_MODEL_DIR)
        with (OUTPUT_DIR / "history_best.json").open("w", encoding="utf-8") as f:
            json.dump(history, f, ensure_ascii=False, indent=2)
        print(f"  New best model saved to {BEST_MODEL_DIR} | BLEU={best_bleu:.2f}")
    else:
        epochs_without_improvement += 1
        print(f"  No significant improvement for {epochs_without_improvement} epoch(s). Best BLEU={best_bleu:.2f}")
        if PATIENCE > 0 and epochs_without_improvement >= PATIENCE:
            print(f"Early stopping triggered (patience={PATIENCE}).")
            break

print(f"Best validation BLEU during training: {best_bleu:.2f}")


Starting from pretrained model: google/byt5-small


Loading weights: 100%|██████████| 172/172 [00:00<00:00, 74220.19it/s]
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.
c:\Users\Администратор\AppData\Local\Programs\Python\Python313\Lib\site-packages\torch\nn\modules\module.py:1367: UserWarning: expandable_segments not supported on this platform (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\c10/cuda/CUDAAllocatorConfig.h:39.)
  return t.to(
Epoch 1/12: 100%|██████████| 75000/75000 [3:44:39<00:00,  5.56it/s, loss=1.4011, lr=9.75e-05, skipped=0]  


Epoch 1: train_loss=1.4011, val_BLEU=0.78
  sample val pred: You know, there was a lot of people in Bordeaux, and he could get away from 27:37.
  sample val gold: The hosts regrouped, and Bouchard evened the score again, scoring a goal with a 27-37 man advantage.


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.25it/s]


  New best model saved to outputs_zetan_byt5_notebook\best_model | BLEU=0.78


...

Epoch 4: train_loss=0.7314, val_BLEU=5.06
  sample val pred: The housewife collected himself and Bouchard's 27:37 gold equals once.
  sample val gold: The hosts regrouped, and Bouchard evened the score again, scoring a goal with a 27-37 man advantage.


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.17it/s]


  New best model saved to outputs_zetan_byt5_notebook\best_model | BLEU=5.06


Epoch 5/12:  12%|█▏        | 9272/75000 [27:25<3:14:23,  5.64it/s, loss=0.7005, lr=6.98e-05, skipped=0]


KeyboardInterrupt: 

In [6]:
# Optional extra final pass on train+val using best checkpoint (often helps hidden test)
if FINAL_FINETUNE_ON_TRAIN_PLUS_VAL:
    print("Starting optional final finetune on train+val...")
    tokenizer = AutoTokenizer.from_pretrained(BEST_MODEL_DIR)
    model = AutoModelForSeq2SeqLM.from_pretrained(BEST_MODEL_DIR).to(device)

    if hasattr(model, "gradient_checkpointing_enable"):
        model.gradient_checkpointing_enable()
    if hasattr(model.config, "use_cache"):
        model.config.use_cache = False

    full_train_rows = train_rows + val_rows
    full_train_loader = DataLoader(
        PairDataset(full_train_rows),
        batch_size=TRAIN_BATCH_SIZE,
        shuffle=True,
        collate_fn=collator.train_collate,
        num_workers=0,
        pin_memory=torch.cuda.is_available(),
    )

    optimizer_ft = torch.optim.AdamW(model.parameters(), lr=FINAL_FINETUNE_LR, weight_decay=WEIGHT_DECAY)
    total_steps_ft = max(1, math.ceil(len(full_train_loader) / GRADIENT_ACCUMULATION_STEPS) * FINAL_FINETUNE_EPOCHS)
    scheduler_ft = get_linear_schedule_with_warmup(optimizer_ft, int(total_steps_ft * 0.05), total_steps_ft)

    use_cuda = torch.cuda.is_available()
    autocast_device = "cuda" if use_cuda else "cpu"
    use_bf16 = bool(use_cuda and hasattr(torch.cuda, "is_bf16_supported") and torch.cuda.is_bf16_supported())
    use_fp16 = bool(use_cuda and not use_bf16)
    autocast_enabled = use_bf16 or use_fp16
    autocast_dtype = torch.bfloat16 if use_bf16 else torch.float16
    scaler_ft = torch.amp.GradScaler(device=autocast_device, enabled=use_fp16)

    for epoch in range(1, FINAL_FINETUNE_EPOCHS + 1):
        model.train()
        optimizer_ft.zero_grad(set_to_none=True)
        pbar = tqdm(full_train_loader, desc=f"Final finetune epoch {epoch}/{FINAL_FINETUNE_EPOCHS}")
        for step, batch in enumerate(pbar, start=1):
            batch = move_to_device(batch, device)
            with torch.amp.autocast(device_type=autocast_device, enabled=autocast_enabled, dtype=autocast_dtype):
                loss = model(**batch).loss / GRADIENT_ACCUMULATION_STEPS

            if not torch.isfinite(loss):
                optimizer_ft.zero_grad(set_to_none=True)
                continue

            scaler_ft.scale(loss).backward()
            if (step % GRADIENT_ACCUMULATION_STEPS == 0) or (step == len(full_train_loader)):
                scaler_ft.unscale_(optimizer_ft)
                torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
                scaler_ft.step(optimizer_ft)
                scaler_ft.update()
                optimizer_ft.zero_grad(set_to_none=True)
                scheduler_ft.step()

            pbar.set_postfix(loss=float(loss.item() * GRADIENT_ACCUMULATION_STEPS), lr=f"{optimizer_ft.param_groups[0]['lr']:.2e}")

    # overwrite best dir with final model for inference
    model.save_pretrained(BEST_MODEL_DIR)
    tokenizer.save_pretrained(BEST_MODEL_DIR)
    print("Final finetune completed and best model directory updated.")
else:
    print("Skipping final finetune on train+val.")


Starting optional final finetune on train+val...


Loading weights: 100%|██████████| 170/170 [00:00<00:00, 12618.02it/s]
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.
Final finetune epoch 1/1:   0%|          | 136/75125 [00:26<4:04:13,  5.12it/s, loss=0.927, lr=1.81e-06]


KeyboardInterrupt: 

In [7]:
# Evaluate best model on validation and pick best decoding parameters
tokenizer = AutoTokenizer.from_pretrained(BEST_MODEL_DIR)
model = AutoModelForSeq2SeqLM.from_pretrained(BEST_MODEL_DIR).to(device)

# Try a few decode configs and pick the best by validation BLEU.
decode_grid = [
    {"num_beams": 4, "length_penalty": 0.9, "repetition_penalty": 1.10, "no_repeat_ngram_size": 3},
    {"num_beams": 6, "length_penalty": 0.9, "repetition_penalty": 1.12, "no_repeat_ngram_size": 3},
    {"num_beams": 8, "length_penalty": 0.9, "repetition_penalty": 1.12, "no_repeat_ngram_size": 3},
    {"num_beams": 8, "length_penalty": 1.0, "repetition_penalty": 1.12, "no_repeat_ngram_size": 3},
]

best_decode_cfg = None
best_decode_bleu = -1.0
best_decode_preds = None

for cfg in decode_grid:
    bleu, preds = evaluate_bleu(
        model=model,
        tokenizer=tokenizer,
        rows=val_rows,
        collator=collator,
        device=device,
        batch_size=EVAL_BATCH_SIZE,
        max_source_length=MAX_SOURCE_LENGTH,
        max_new_tokens=MAX_NEW_TOKENS,
        num_beams=cfg["num_beams"],
        length_penalty=cfg["length_penalty"],
        repetition_penalty=cfg["repetition_penalty"],
        no_repeat_ngram_size=cfg["no_repeat_ngram_size"],
    )
    print(f"cfg={cfg} -> val BLEU={bleu:.2f}")
    if bleu > best_decode_bleu:
        best_decode_bleu = bleu
        best_decode_cfg = cfg
        best_decode_preds = preds

print(f"Best decode config: {best_decode_cfg} | val BLEU={best_decode_bleu:.2f}")
for i in range(3):
    print(f"[{i}] SRC: {val_rows[i]['src']}")
    print(f"[{i}] PRED: {best_decode_preds[i]}")
    print(f"[{i}] GOLD: {val_rows[i]['dst']}")
    print()


Loading weights: 100%|██████████| 170/170 [00:00<00:00, 11572.56it/s]
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


KeyboardInterrupt: 

In [8]:
# Generate test translations and save submission in required JSONL format

if "best_decode_cfg" not in globals() or best_decode_cfg is None:
    best_decode_cfg = {
        "num_beams": FINAL_NUM_BEAMS,
        "length_penalty": LENGTH_PENALTY,
        "repetition_penalty": REPETITION_PENALTY,
        "no_repeat_ngram_size": NO_REPEAT_NGRAM_SIZE,
    }
    print("best_decode_cfg was not set; using default decode config:", best_decode_cfg)

test_loader = DataLoader(
    PairDataset(test_rows),
    batch_size=PREDICT_BATCH_SIZE,
    shuffle=False,
    collate_fn=collator.infer_collate,
)

test_preds = generate_texts(
    model=model,
    tokenizer=tokenizer,
    dataloader=test_loader,
    device=device,
    max_source_length=MAX_SOURCE_LENGTH,
    max_new_tokens=MAX_NEW_TOKENS,
    num_beams=best_decode_cfg["num_beams"],
    length_penalty=best_decode_cfg["length_penalty"],
    repetition_penalty=best_decode_cfg["repetition_penalty"],
    no_repeat_ngram_size=best_decode_cfg["no_repeat_ngram_size"],
)

with SUBMISSION_PATH.open("w", encoding="utf-8") as f:
    for row, pred in zip(test_rows, test_preds):
        f.write(json.dumps({"src": row["src"], "dst": pred.strip()}, ensure_ascii=False) + "\n")

print(f"Submission saved to: {SUBMISSION_PATH.resolve()}")
print(f"Total lines: {len(test_preds)}")
print("Sample outputs:")
for i in range(3):
    print(f"[{i}] src: {test_rows[i]['src']}")
    print(f"[{i}] dst: {test_preds[i]}")


TypeError: 'NoneType' object is not subscriptable